# Setup

In [2]:
import time
inital_start = time.time()
import subprocess
from pathlib import Path
import pandas as pd
import gzip
from glob import glob
import os
import psutil

logical_cpus = psutil.cpu_count(logical=True)
physical_cores = psutil.cpu_count(logical=False)

threads_per_job = max(1, logical_cpus // physical_cores)

print(physical_cores)
print(threads_per_job)

# 1. Load in weights file from PGS Catalog

In [3]:
##### Input variables for specific analysis #####
PGS_ID = "PGS002308"
BUILD = "GRCh38" #AoU data is on GRCh38 so this is desired
#################################################

##### Load in PGS score file from PGS catalog #####
url = f"https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/{PGS_ID}/ScoringFiles/Harmonized/{PGS_ID}_hmPOS_{BUILD}.txt.gz"
raw_file = Path(f"{PGS_ID}_hmPOS_{BUILD}.txt.gz")

# Save locally 
file_destination = "gs://rw-migration-aou-rw-ecba6cc3/pgs_weights/"
subprocess.run(["gcloud", "storage", "cp", str(raw_file), file_destination], check=True)

# Read into enviornment 
score_df = pd.read_csv(
    raw_file,
    sep="\t",
    comment="#",
    compression="gzip"
)

score_df.head()

Copying file://PGS002308_hmPOS_GRCh38.txt.gz to gs://rw-migration-aou-rw-ecba6cc3/pgs_weights/PGS002308_hmPOS_GRCh38.txt.gz
  
..

Average throughput: 112.8MiB/s
/tmp/ipykernel_1750/4041334530.py:11: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  score_df = pd.read_csv(


,rsID,chr_name,chr_position,effect_allele,other_allele,effect_weight,hm_source,hm_rsID,hm_chr,hm_pos,hm_inferOtherAllele
0,rs3131972,1,752721,A,G,-0.000114,ENSEMBL,rs3131972,1.0,817341.0,NaN
1,rs3131969,1,754182,A,G,-0.000142,ENSEMBL,rs3131969,1.0,818802.0,NaN
2,rs1048488,1,760912,C,T,0.000005,ENSEMBL,rs1048488,1.0,825532.0,NaN
3,rs12562034,1,768448,A,G,0.000011,ENSEMBL,rs12562034,1.0,833068.0,NaN
4,rs4040617,1,779322,G,A,-0.000060,ENSEMBL,rs4040617,1.0,843942.0,NaN


# 2. Liftover score file to GRCh38 as needed

In [4]:
start = time.time()

##### Extract genome build of score file from meta-data #####
# Although the file name says GRCh38, this is not always true
# and the meta-data is more reliable. 

genome_build = None

with gzip.open(raw_file, "rt") as f:
    for line in f:
        if line.startswith("#genome_build="):
            genome_build = line.strip().split("=")[1]
            break

print("Genome build:", genome_build)

##### Perform liftover if needed #####

needs_liftover = genome_build in ["GRCh37", "hg19"]

if needs_liftover:
    !pip install pyliftover

    print("Lifting over from ", genome_build, "to GRCh38")

    from pyliftover import LiftOver
    lo = LiftOver("hg19", "hg38")

    def liftover_pos(chrom, pos):
        result = lo.convert_coordinate(f"chr{chrom}", int(pos))
        if result:
            new_chr, new_pos, _, _ = result[0]
            return new_chr.replace("chr", ""), new_pos
        return None, None

    score_df[["chr_name_38", "chr_position_38"]] = score_df.apply(
        lambda row: liftover_pos(row["chr_name"], row["chr_position"]),
        axis=1,
        result_type="expand"
    )

    score_df["chr_position_38"] = pd.to_numeric(
        score_df["chr_position_38"],
        errors="coerce"
    ).astype("Int64")

else:
    print("No liftover needed, already in GRCh38")
    # No liftover needed — just copy original coordinates
    score_df["chr_name_38"] = score_df["chr_name"]
    score_df["chr_position_38"] = pd.to_numeric(
        score_df["chr_position"],
        errors="coerce"
    ).astype("Int64")

print(f"Done ({time.time()-start:.1f}s)")


Genome build: hg19
Lifting over from  hg19 to GRCh38
Done (50.3s)


# 3. Prepare SNP IDs to match .bim file

In [5]:
bim_start = time.time()
start = time.time()
print("Starting harmonization...")

###########################################################################################
#Grab BIM IDs
###########################################################################################

# Load all BIM files
print("Finding BIM files...")
bim_files = glob("/home/jupyter/workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr*.bim")

##### track time #####
print(f"Found {len(bim_files)} BIM files ({time.time()-start:.1f}s)")
######################

# Combine bim files into single file
print("Loading BIM files...")
bim = pd.concat(
    [pd.read_csv(f, sep="\t", header=None) for f in bim_files],
    ignore_index=True
)

# Set of BIM SNP IDs for fast lookup
bim_ids = set(bim.iloc[:,1])

##### track time #####
print(f"Loaded {len(bim):,} variants ({time.time()-start:.1f}s)")
######################

##########################################################################################
##Build forward and reverse IDs from PGS catalog score file
##########################################################################################

# Build forward and reverse IDs
print("Building SNP IDs...")
score_df['SNP_ID'] = (
    "chr" + score_df['chr_name'].astype(str)
    + ":" + score_df['chr_position_38'].astype(str)
    + ":" + score_df['effect_allele']
    + ":" + score_df['other_allele']
)

score_df['SNP_rev'] = (
    "chr" + score_df['chr_name'].astype(str)
    + ":" + score_df['chr_position_38'].astype(str)
    + ":" + score_df['other_allele']
    + ":" + score_df['effect_allele']
)

##### track time #####
print(f"Done ({time.time()-start:.1f}s)")
######################

#########################################################################################
### Find allele matches from PGS score file in BIM file
##########################################################################################

# Determine matches
print("Determining matches...")
forward_match = score_df['SNP_ID'].isin(bim_ids)
reverse_match = score_df['SNP_rev'].isin(bim_ids)
both_match = forward_match & reverse_match
flip_mask = (~forward_match) & reverse_match

#########################
# Print statements for QC tracking
#########################

print(f"Forward matches: {forward_match.sum()}")
print(f"Reverse matches: {reverse_match.sum()}")
print(f"Both orientations (Sanity check - should be 0): {both_match.sum()}") 

print(f"Total PRS variants: {len(score_df):,}")
print(f"Total matched variants: {(forward_match | reverse_match).sum()}")

n_found = (forward_match | reverse_match).sum()
n_total = len(score_df)
print(f"Variants found: {n_found:,}/{n_total:,}")
print(f"Percent found: {100 * n_found / n_total:.2f}%")

##### track time #####
print(f"Matches found ({time.time()-start:.1f}s)")
######################

##########################################################################################
#### Create harmonized score file
##########################################################################################
score_df_final = score_df.copy()

# Flip effect weights where necessary
print("Applying allele flips...")
score_df_final.loc[flip_mask, 'effect_weight'] *= -1

# Swap alleles
score_df_final.loc[flip_mask, 'effect_allele'] = score_df.loc[flip_mask, 'other_allele']
score_df_final.loc[flip_mask, 'other_allele'] = score_df.loc[flip_mask, 'effect_allele']

##### track time #####
print(f"Flips applied ({time.time()-start:.1f}s)")
######################

# Use reversed SNP IDs for flipped variants
score_df_final.loc[flip_mask, 'SNP_ID'] = score_df.loc[flip_mask, 'SNP_rev']

n_total = score_df_final['SNP_ID'].isin(bim_ids).sum() 

print(f"Total orientation matches: {n_total}")

#########################################################################################
##### Save new PRS file
##########################################################################################
# Keep only matched variants
score_df_final = score_df_final[forward_match | reverse_match]

# Select columns for PLINK score file
plink_score_df = score_df_final[['SNP_ID', 'effect_allele', 'effect_weight']]

# Save
plink_score_file = f"/home/jupyter/workspace/rw-migration-aou-rw-ecba6cc3/pgs_weights/{PGS_ID}_plink_score_{BUILD}_prepared.txt"
plink_score_df.to_csv(
    plink_score_file,
    sep="\t",
    index=False,
    header=False
)

print(f"Saved {len(plink_score_df):,} variants to:")
print(plink_score_file)

##### track time #####
print(f"Done; Total time: {(time.time() - bim_start)/60:.1f} minutes")
######################

Starting harmonization...
Finding BIM files...
Found 24 BIM files (0.3s)
Loading BIM files...
Loaded 111,404,689 variants (184.7s)
Building SNP IDs...
Done (188.0s)
Determining matches...
Forward matches: 346731
Reverse matches: 911555
Both orientations: 0
Total PRS variants: 1,259,754
Total matched variants: 1258286
Variants found: 1,258,286/1,259,754
Percent found: 99.88%
Matches found (644.1s)
Applying allele flips...
Flips applied (644.9s)
Total orientation matches: 1258286


NameError: name 'plink_score_file' is not defined

# 4. Run Plink

## Generate script file

In [7]:
%%writefile ~/plink_bed.sh
#!/bin/bash

set -o pipefail
set -o errexit

SECONDS=0

INPUT_PATH="/home/jupyter/workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed"
OUTPUT_BASE="/home/jupyter/workspace/rw-migration-aou-rw-ecba6cc3/PRS_scoring_using_bed_files"
WEIGHTS_PATH="/home/jupyter/workspace/rw-migration-aou-rw-ecba6cc3/pgs_weights"

MAX_JOBS="${MAX_JOBS:-1}"
PLINK_THREADS="${PLINK_THREADS:-1}"

RUN_ID="$(date +%Y%m%d_%H%M%S)"
OUTPUT_PATH="${OUTPUT_BASE}/${PGS}_${BUILD}_${RUN_ID}"

mkdir -p "${OUTPUT_PATH}"

echo "Output path is ${OUTPUT_PATH}"
echo "MAX_JOBS=${MAX_JOBS}"
echo "PLINK_THREADS=${PLINK_THREADS}"

for chrom in ${CHROMS}; do
    (
        chr_start=$SECONDS
        echo "Chromosome: ${chrom}"

        bed_prefix="${INPUT_PATH}/chr${chrom}"
        score_file="${WEIGHTS_PATH}/${PGS}_plink_score_${BUILD}_prepared.txt"
        out_prefix="${OUTPUT_PATH}/${PGS}_score_file_chr${chrom}"

        plink \
            --bfile "${bed_prefix}" \
            --score "${score_file}" 1 2 3 header \
            --threads "${PLINK_THREADS}" \
            --out "${out_prefix}"

        chr_elapsed=$((SECONDS - chr_start))
        printf "Chromosome %s complete in %02d:%02d:%02d\n" \
            "${chrom}" \
            $((chr_elapsed / 3600)) \
            $(((chr_elapsed % 3600) / 60)) \
            $((chr_elapsed % 60))
    ) &

    while [[ $(jobs -r -p | wc -l) -ge ${MAX_JOBS} ]]; do
        sleep 2
    done
done

wait

elapsed=$SECONDS
echo "Finished at $(date)"
printf "All chromosome jobs complete in %02d:%02d:%02d\n" \
    $((elapsed / 3600)) \
    $(((elapsed % 3600) / 60)) \
    $((elapsed % 60))

Writing /home/jupyter/plink_bed.sh


## Execute script

In [ ]:
plink_start = time.time()

env = os.environ.copy()
env.update({
    "PGS": PGS_ID,
    "BUILD": BUILD,
    "CHROMS": "21 22",
    "MAX_JOBS": str(physical_cores),
    "PLINK_THREADS": str(threads_per_job),
})

subprocess.run(["chmod", "+x", "/home/jupyter/plink_bed.sh"], check=True)
subprocess.run(["/home/jupyter/plink_bed.sh"], check=True, env=env)

print(f"Done; Total time: {(time.time() - plink_start)/60:.1f} minutes")

Output path is /home/jupyter/workspace/rw-migration-aou-rw-ecba6cc3/PRS_scoring_using_bed_files/PGS002308_GRCh38_20260624_142952
MAX_JOBS=4
PLINK_THREADS=2
Chromosome: 21
Chromosome: 22
PLINK v1.9.0-b.8 64-bit (22 Oct 2024)              cog-genomics.org/plink/1.9/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/rw-migration-aou-rw-ecba6cc3/PRS_scoring_using_bed_files/PGS002308_GRCh38_20260624_142952/PGS002308_score_file_chr21.log.
Options in effect:
  --bfile /home/jupyter/workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr21
  --out /home/jupyter/workspace/rw-migration-aou-rw-ecba6cc3/PRS_scoring_using_bed_files/PGS002308_GRCh38_20260624_142952/PGS002308_score_file_chr21
  --score /home/jupyter/workspace/rw-migration-aou-rw-ecba6cc3/pgs_weights/PGS002308_plink_score_GRCh38_prepared.txt 1 2 3 header
  --threads 2

52195 MB RAM detected; reserving 26097 MB for main workspace.
1672

# 5. Combine Chromosomes 

# 6. Check QC logs 

# 7. Perform ancestry adjustment